# Trader AI Dashboard v1 - Colab Launcher

Run the cells below to download the project, fetch market data, and display the v1 dashboard.

In [ ]:
%pip -q install yfinance pandas numpy matplotlib

In [ ]:
from pathlib import Path
import subprocess, sys

# Direct GitHub-to-Colab launches only open this file, so fetch src/ when needed.
root = Path('/content/trader-ai-dashboard')
if not (root / 'src').exists():
    subprocess.run(['git', 'clone', 'https://github.com/cottonlyz-coder/trader-ai-dashboard.git', str(root)], check=True)
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from src.data_loader import load_market_data
from src.indicators import calculate_market_metrics, normalized_prices
from src.regime import classify_market_regime
from src.report import generate_trader_commentary

prices, symbols, messages = load_market_data(period='6mo')
metrics = calculate_market_metrics(prices)
regime = classify_market_regime(metrics)

display(Markdown('## Data Series Used'))
display(symbols)
if messages:
    display(Markdown('## Data Notes'))
    for message in messages:
        print('-', message)

display(Markdown('## Market Metrics'))
display(metrics.style.format({'Latest Price': '{:,.2f}', '1D Change (%)': '{:+.2f}%', '5D Change (%)': '{:+.2f}%', '20D Change (%)': '{:+.2f}%', '20D Volatility (%)': '{:.2f}%'}))
display(Markdown(f'## Market Regime: **{regime.label}** (score: {regime.score:+d})'))

normalized_prices(prices).plot(figsize=(14, 7), linewidth=1.8, title='Normalized Market Performance (Start = 100)')
plt.axhline(100, color='grey', linewidth=1, linestyle='--')
plt.ylabel('Normalized Price')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

display(Markdown('## Trader Commentary'))
display(Markdown(generate_trader_commentary(metrics, regime)))